<a href="https://colab.research.google.com/github/CHO-max1234/lettuce-project-in-Vertical-farm/blob/main/260426_%EC%A4%91%EA%B0%84_%EC%A0%90%EA%B2%80_%EC%8B%9C%EA%B0%81%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
2작기 정면 crop 이미지 지표 추출 스크립트
- 입력: crop 이미지 폴더 + slot_crop_meta CSV + scale_map CSV
- 출력: 36개 지표 CSV
- 병렬 처리 (multiprocessing.Pool)
- 경로는 아래 ★ 표시 부분만 수정
"""

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.filters import sato
from multiprocessing import Pool, cpu_count
import warnings
import re
import time

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════
# ★ 경로 설정 (여기만 수정)
# ════════════════════════════════════════════════════════
CROP_DIR      = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/2작기_lettuce/이름변경 전_to 0412/0. crops_to260412/crops")   # crop 이미지 폴더 경로
META_CSV      = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/2작기_lettuce/이름변경 전_to 0412/0. crops_to260412/260414_slot_crop_meta_all.csv")   # 260414 또는 260425 slot_crop_meta CSV
SCALE_CSV     = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/4. scale/scale_map_be0412.csv")   # scale_map_af0412.csv
OUT_CSV       = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 시각화/~0412 중간점검/be260412_metrics.csv")   # 결과 저장 경로 (예: 260425_metrics.csv)

# ★ Sato roi_r 계산 방식 (필요시 조정)
# 'avg'  : (bbox_w + bbox_h) / 4  — 기본값, b슬롯 가로 넓은 경우 대응
# 'min'  : min(bbox_w, bbox_h) / 2
# 'max'  : max(bbox_w, bbox_h) / 2
ROI_R_MODE = 'avg'

# ★ 병렬 처리 워커 수 (기본: CPU 수 - 1)
N_WORKERS = max(1, cpu_count() - 1)

# ★ 청크 크기 (진행률 표시 단위)
CHUNK_SIZE = 500
# ════════════════════════════════════════════════════════


In [2]:

# ────────────────────────────────────────────────────────
# 함수: 마스크 생성 (검정 배경 제거)
#   crop 이미지는 검정 배경 + 상추만 있으므로 threshold로 충분
# ────────────────────────────────────────────────────────
def get_mask(gray, threshold=10):
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    # 노이즈 제거: 작은 연결 성분 제거
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    return mask


# ────────────────────────────────────────────────────────
# 함수: roi_r 계산
# ────────────────────────────────────────────────────────
def get_roi_r(bbox_w, bbox_h, mode=ROI_R_MODE):
    if mode == 'avg':
        return (bbox_w + bbox_h) / 4
    elif mode == 'min':
        return min(bbox_w, bbox_h) / 2
    elif mode == 'max':
        return max(bbox_w, bbox_h) / 2
    return (bbox_w + bbox_h) / 4


# ────────────────────────────────────────────────────────
# 함수: Contour 기반 지표 (B그룹)
# ────────────────────────────────────────────────────────
def compute_contour_metrics(gray, mask, mm_per_px):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return {k: np.nan for k in [
            'roi_area_px', 'area_cm2', 'perimeter_px',
            'circularity', 'solidity', 'curvature', 'aspect_ratio', 'bottom_flatness'
        ]}
    cnt = max(contours, key=cv2.contourArea)

    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)

    circularity = float(4 * np.pi * area / (peri ** 2)) if peri > 0 else np.nan
    solidity    = float(area / hull_area) if hull_area > 0 else np.nan

    # curvature: contour 50점 샘플링 → 인접 벡터 각도 변화량 평균
    pts = cnt[:, 0, :]
    n   = len(pts)
    if n >= 10:
        idx   = np.linspace(0, n - 1, 50, dtype=int)
        spts  = pts[idx]
        v1    = spts[1:] - spts[:-1]
        v2    = spts[2:] - spts[1:-1]
        dot   = np.sum(v1[:-1] * v2, axis=1)
        norm  = np.linalg.norm(v1[:-1], axis=1) * np.linalg.norm(v2, axis=1)
        cos_a = np.clip(dot / (norm + 1e-8), -1, 1)
        curvature = float(np.mean(np.arccos(cos_a)))
    else:
        curvature = np.nan

    # aspect_ratio
    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = float(w / h) if h > 0 else np.nan

    # bottom_flatness: 하단 5px 이내 contour 점들의 x 범위 / bbox_w
    bottom_pts = pts[pts[:, 1] >= (y + h - 5)]
    if len(bottom_pts) >= 2 and w > 0:
        bottom_flatness = float((bottom_pts[:, 0].max() - bottom_pts[:, 0].min()) / w)
    else:
        bottom_flatness = np.nan

    # area_cm2
    if not np.isnan(mm_per_px) and mm_per_px > 0:
        area_cm2 = float(area * (mm_per_px / 10) ** 2)
    else:
        area_cm2 = np.nan

    return {
        'roi_area_px':     float(area),
        'area_cm2':        area_cm2,
        'perimeter_px':    float(peri),
        'circularity':     circularity,
        'solidity':        solidity,
        'curvature':       curvature,
        'aspect_ratio':    aspect_ratio,
        'bottom_flatness': bottom_flatness,
    }


# ────────────────────────────────────────────────────────
# 함수: 상하 분리 지표 (C그룹)
#   cy_crop: crop 이미지 내부 기준 중심 y 좌표
# ────────────────────────────────────────────────────────
def compute_upper_lower_metrics(gray, mask, cy_crop):
    h_img = mask.shape[0]
    cy_int = int(np.clip(cy_crop, 1, h_img - 1))

    upper_mask = mask.copy(); upper_mask[cy_int:, :] = 0
    lower_mask = mask.copy(); lower_mask[:cy_int, :] = 0

    def area_of(m):
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cnts:
            return 0.0
        return float(cv2.contourArea(max(cnts, key=cv2.contourArea)))

    def circularity_of(m):
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cnts:
            return np.nan
        cnt = max(cnts, key=cv2.contourArea)
        a = cv2.contourArea(cnt)
        p = cv2.arcLength(cnt, True)
        return float(4 * np.pi * a / (p ** 2)) if p > 0 else np.nan

    upper_area = area_of(upper_mask)
    lower_area = area_of(lower_mask)
    ratio      = float(upper_area / lower_area) if lower_area > 0 else np.nan
    upper_circ = circularity_of(upper_mask)

    # upper_edge_density: 상단 Canny 엣지 픽셀 수 / 상단 마스크 픽셀 수
    upper_gray = gray.copy(); upper_gray[cy_int:, :] = 0
    edges      = cv2.Canny(upper_gray, 50, 150)
    upper_px   = float(np.sum(upper_mask > 0))
    upper_edge_density = float(np.sum(edges > 0) / upper_px) if upper_px > 0 else np.nan

    return {
        'upper_area_px':          upper_area,
        'lower_area_px':          lower_area,
        'upper_lower_area_ratio': ratio,
        'upper_circularity':      upper_circ,
        'upper_edge_density':     upper_edge_density,
    }


# ────────────────────────────────────────────────────────
# 함수: 방사형 지표 (D그룹)
#   cx_crop, cy_crop: crop 이미지 내부 기준 좌표
# ────────────────────────────────────────────────────────
def compute_radial_metrics(mask, cx_crop, cy_crop):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return {'radial_min': np.nan, 'radial_max_min_ratio': np.nan}
    cnt  = max(contours, key=cv2.contourArea)
    pts  = cnt[:, 0, :].astype(float)
    dists = np.sqrt((pts[:, 0] - cx_crop) ** 2 + (pts[:, 1] - cy_crop) ** 2)
    r_min = float(dists.min())
    r_max = float(dists.max())
    ratio = float(r_max / r_min) if r_min > 0 else np.nan
    return {
        'radial_min':           r_min,
        'radial_max_min_ratio': ratio,
    }


# ────────────────────────────────────────────────────────
# 함수: 밝기/질감 지표 (E그룹)
# ────────────────────────────────────────────────────────
def compute_texture_metrics(gray, mask):
    px = gray[mask > 0]
    if len(px) == 0:
        return {'brightness_mean': np.nan, 'dark_ratio': np.nan, 'grad_std': np.nan}

    brightness_mean = float(px.mean())
    dark_ratio      = float((px < 80).sum() / len(px))

    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx ** 2 + sy ** 2)
    grad_std = float(mag[mask > 0].std())

    return {
        'brightness_mean': brightness_mean,
        'dark_ratio':      dark_ratio,
        'grad_std':        grad_std,
    }


# ────────────────────────────────────────────────────────
# 함수: Sato-ridge 지표 (F그룹)
#   cx_crop, cy_crop: crop 이미지 내부 기준 좌표
#   roi_r: 분석 반경 (bbox 기준 계산)
#   suffix: '' (전체 crop) 또는 '_core' (core crop, 결구 후)
# ────────────────────────────────────────────────────────
def compute_ridge_metrics(gray, cx_crop, cy_crop, roi_r, threshold_pct=70, suffix=''):
    nan_keys = [f'inner_ratio_30{suffix}', f'inner_ratio_50{suffix}',
                f'ridge_circularity{suffix}', f'ridge_std_r{suffix}',
                f'ridge_mean_r{suffix}', f'angular_cv{suffix}']
    if roi_r <= 0:
        return {k: np.nan for k in nan_keys}

    h, w = gray.shape
    img_f    = gray.astype(np.float64) / 255.0
    sato_map = sato(img_f, sigmas=range(1, 8), black_ridges=True, mode='reflect')

    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(roi_mask, (int(cx_crop), int(cy_crop)), int(roi_r), 255, -1)

    sato_roi  = sato_map.copy()
    sato_roi[roi_mask == 0] = 0
    valid_all = sato_roi[roi_mask > 0]

    if len(valid_all) == 0:
        return {k: np.nan for k in nan_keys}

    thr    = np.percentile(valid_all, threshold_pct)
    ys, xs = np.where((sato_roi > thr) & (roi_mask > 0))
    n_ridge = len(xs)

    if n_ridge < 10:
        return {k: np.nan for k in nan_keys}

    radii = np.sqrt((xs - cx_crop) ** 2 + (ys - cy_crop) ** 2)

    ridge_mean_r      = float(np.mean(radii) / roi_r)
    ridge_std_r       = float(np.std(radii) / roi_r)
    inner_ratio_30    = float(np.sum(radii < 0.3 * roi_r) / n_ridge)
    inner_ratio_50    = float(np.sum(radii < 0.5 * roi_r) / n_ridge)
    angles            = np.arctan2(ys - cy_crop, xs - cx_crop)
    bins              = np.linspace(-np.pi, np.pi, 9)
    counts, _         = np.histogram(angles, bins=bins)
    angular_cv        = float(np.std(counts) / np.mean(counts)) if np.mean(counts) > 0 else np.nan
    ridge_circularity = float(1.0 - (np.std(radii) / np.mean(radii))) if np.mean(radii) > 0 else np.nan

    return {
        f'inner_ratio_30{suffix}':    round(inner_ratio_30, 4),
        f'inner_ratio_50{suffix}':    round(inner_ratio_50, 4),
        f'ridge_circularity{suffix}': round(ridge_circularity, 4),
        f'ridge_std_r{suffix}':       round(ridge_std_r, 4),
        f'ridge_mean_r{suffix}':      round(ridge_mean_r, 4),
        f'angular_cv{suffix}':        round(angular_cv, 4),
    }


# ────────────────────────────────────────────────────────
# 함수: blur_score
# ────────────────────────────────────────────────────────
def compute_blur_score(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


# ────────────────────────────────────────────────────────
# 함수: crop 이미지 파일 찾기
#   CROP_DIR 하위에 bed별 폴더가 있을 수 있으므로 재귀 탐색
# ────────────────────────────────────────────────────────
def find_crop_file(crop_dir, fname):
    # 1순위: 직접 경로
    p = Path(crop_dir) / fname
    if p.exists():
        return p
    # 2순위: 하위 폴더 재귀 탐색
    matches = list(Path(crop_dir).rglob(fname))
    if matches:
        return matches[0]
    return None


# ────────────────────────────────────────────────────────
# 함수: 단일 이미지 처리 (병렬 처리 단위)
# ────────────────────────────────────────────────────────
def process_one(args):
    row, crop_dir = args
    fname = str(row['crop_filename'])

    # 날짜 파싱
    m = re.search(r'(\d{8})', fname)
    date_str = m.group(1) if m else np.nan

    result = {
        'crop_filename': fname,
        'bed_name':      row.get('bed_name', np.nan),
        'slot_name':     row.get('slot_name', np.nan),
        'date':          date_str,
        'conf':          row.get('conf', np.nan),
    }

    # NaN 채움용 키 목록
    all_metric_keys = [
        'blur_score',
        'roi_area_px', 'area_cm2', 'perimeter_px',
        'circularity', 'solidity', 'curvature', 'aspect_ratio', 'bottom_flatness',
        'upper_area_px', 'lower_area_px', 'upper_lower_area_ratio',
        'upper_circularity', 'upper_edge_density',
        'radial_min', 'radial_max_min_ratio',
        'brightness_mean', 'dark_ratio', 'grad_std',
        'inner_ratio_30', 'inner_ratio_50', 'ridge_circularity',
        'ridge_std_r', 'ridge_mean_r', 'angular_cv',
        'inner_ratio_30_core', 'inner_ratio_50_core', 'ridge_circularity_core',
        'ridge_std_r_core', 'ridge_mean_r_core', 'angular_cv_core',
    ]

    # 이미지 파일 탐색
    img_path = find_crop_file(crop_dir, fname)
    if img_path is None:
        for k in all_metric_keys:
            result[k] = np.nan
        return result

    # 이미지 로드
    img = cv2.imread(str(img_path))
    if img is None:
        for k in all_metric_keys:
            result[k] = np.nan
        return result

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = get_mask(gray)

    # cx, cy는 이미 crop 이미지 내부 기준 좌표 (상추 1개의 중심)
    cx_crop   = float(row.get('cx', img.shape[1] / 2))
    cy_crop   = float(row.get('cy', img.shape[0] / 2))
    bbox_w    = float(row.get('bbox_w', img.shape[1]))
    bbox_h    = float(row.get('bbox_h', img.shape[0]))
    mm_per_px = float(row.get('mm_per_px', np.nan))

    # crop 이미지 범위 클리핑 (안전장치)
    h_img, w_img = gray.shape
    cx_crop = float(np.clip(cx_crop, 1, w_img - 1))
    cy_crop = float(np.clip(cy_crop, 1, h_img - 1))

    roi_r = get_roi_r(bbox_w, bbox_h)

    # blur_score
    result['blur_score'] = compute_blur_score(gray)

    # B그룹: Contour
    result.update(compute_contour_metrics(gray, mask, mm_per_px))

    # C그룹: 상하 분리
    result.update(compute_upper_lower_metrics(gray, mask, cy_crop))

    # D그룹: 방사형
    result.update(compute_radial_metrics(mask, cx_crop, cy_crop))

    # E그룹: 밝기/질감
    result.update(compute_texture_metrics(gray, mask))

    # F그룹: Sato-ridge (전체 crop 기준)
    result.update(compute_ridge_metrics(gray, cx_crop, cy_crop, roi_r, suffix=''))

    # F_core 그룹: 결구 전 → NaN (결구 후 core crop 생기면 채울 것)
    for k in ['inner_ratio_30_core', 'inner_ratio_50_core', 'ridge_circularity_core',
              'ridge_std_r_core', 'ridge_mean_r_core', 'angular_cv_core']:
        result[k] = np.nan

    return result


# ════════════════════════════════════════════════════════
# 메인 처리
# ════════════════════════════════════════════════════════
if __name__ == '__main__':
    t0 = time.time()

    # 1. CSV 로드
    print("[1/4] CSV 로드 중...")
    df_meta  = pd.read_csv(META_CSV)
    df_scale = pd.read_csv(SCALE_CSV)

    # 2. scale_map 병합 (key: bed번호_날짜_시간_cam번호)
    print("[2/4] scale_map 병합 중...")
    df_scale['key'] = df_scale['image_name'].str.extract(r'(bed\d+_\d{8}_\d{6}_cam\d+)')
    df_meta['key']  = df_meta['crop_filename'].str.extract(r'(bed\d+_\d{8}_\d{6}_cam\d+)')
    df_ok   = df_scale[df_scale['qc'] == 'ok'][['key', 'mm_per_px']]
    df_meta = df_meta.merge(df_ok, on='key', how='left')
    print(f"    총 crop: {len(df_meta)}, scale 매칭: {df_meta['mm_per_px'].notna().sum()}")

    # 3. ok crop만 필터
    if 'status' in df_meta.columns:
        df_meta = df_meta[df_meta['status'] == 'ok'].reset_index(drop=True)
    if 'part_no' in df_meta.columns:
        df_meta = df_meta[df_meta['part_no'] == 1].reset_index(drop=True)
    print(f"    필터 후 crop: {len(df_meta)}")

    # 4. 병렬 처리
    print(f"[3/4] 지표 추출 시작 (워커: {N_WORKERS}, 청크: {CHUNK_SIZE})...")
    args_list = [(row.to_dict(), str(CROP_DIR)) for _, row in df_meta.iterrows()]
    results   = []

    with Pool(N_WORKERS) as pool:
        for i, res in enumerate(pool.imap(process_one, args_list, chunksize=50)):
            results.append(res)
            if (i + 1) % CHUNK_SIZE == 0:
                elapsed = time.time() - t0
                remain  = elapsed / (i + 1) * (len(args_list) - i - 1)
                print(f"    {i+1}/{len(args_list)} 완료 | 경과: {elapsed:.0f}s | 잔여: {remain:.0f}s")

    # 5. 저장
    print("[4/4] 결과 저장 중...")
    df_out = pd.DataFrame(results)

    col_order = [
        'crop_filename', 'bed_name', 'slot_name', 'date', 'conf', 'blur_score',
        'roi_area_px', 'area_cm2', 'perimeter_px', 'circularity', 'solidity',
        'curvature', 'aspect_ratio', 'bottom_flatness',
        'upper_area_px', 'lower_area_px', 'upper_lower_area_ratio',
        'upper_circularity', 'upper_edge_density',
        'radial_min', 'radial_max_min_ratio',
        'brightness_mean', 'dark_ratio', 'grad_std',
        'inner_ratio_30', 'inner_ratio_50', 'ridge_circularity',
        'ridge_std_r', 'ridge_mean_r', 'angular_cv',
        'inner_ratio_30_core', 'inner_ratio_50_core', 'ridge_circularity_core',
        'ridge_std_r_core', 'ridge_mean_r_core', 'angular_cv_core',
    ]
    df_out = df_out[[c for c in col_order if c in df_out.columns]]
    df_out.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

    total = time.time() - t0
    print(f"\n완료! 총 {len(df_out)}행 저장: {OUT_CSV}")
    print(f"소요 시간: {total:.0f}초 ({total/60:.1f}분)")

[1/4] CSV 로드 중...
[2/4] scale_map 병합 중...
    총 crop: 14162, scale 매칭: 12174
    필터 후 crop: 11978
[3/4] 지표 추출 시작 (워커: 1, 청크: 500)...
    500/11978 완료 | 경과: 478s | 잔여: 10974s
    1000/11978 완료 | 경과: 930s | 잔여: 10205s
    1500/11978 완료 | 경과: 1390s | 잔여: 9709s
    2000/11978 완료 | 경과: 1845s | 잔여: 9204s
    2500/11978 완료 | 경과: 2302s | 잔여: 8728s
    3000/11978 완료 | 경과: 2769s | 잔여: 8287s
    3500/11978 완료 | 경과: 3255s | 잔여: 7884s
    4000/11978 완료 | 경과: 3736s | 잔여: 7452s
    4500/11978 완료 | 경과: 4206s | 잔여: 6990s
    5000/11978 완료 | 경과: 4707s | 잔여: 6569s
    5500/11978 완료 | 경과: 5150s | 잔여: 6066s
    6000/11978 완료 | 경과: 5588s | 잔여: 5568s
    6500/11978 완료 | 경과: 6033s | 잔여: 5085s
    7000/11978 완료 | 경과: 6503s | 잔여: 4625s
    7500/11978 완료 | 경과: 6952s | 잔여: 4151s
    8000/11978 완료 | 경과: 7418s | 잔여: 3688s
    8500/11978 완료 | 경과: 7893s | 잔여: 3230s
    9000/11978 완료 | 경과: 8363s | 잔여: 2767s
    9500/11978 완료 | 경과: 8873s | 잔여: 2314s
    10000/11978 완료 | 경과: 9376s | 잔여: 1855s
    10500/11978 완료 | 경과: 98